# Vector compression survey + token embedding application

**Part 1 (survey):** JL projections, rank-k truncation, sign/scalar quantization — see `docs/SURVEY.md`.

**Part 3 (this notebook):** GloVe token vectors as a stand-in for embedding-table / KV-cache keys; measure recall@k under compression.

Inspired by [TurboQuant](https://research.google/blog/turboquant-redefining-ai-efficiency-with-extreme-compression/) (pedagogical demo, not a reproduction).

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

import polars as pl
from IPython.display import display, Markdown

REPO = Path.cwd().resolve()
if not (REPO / "scripts" / "run_all.py").exists():
    REPO = REPO.parents[1]
sys.path.insert(0, str(REPO / "python" / "src"))

from vector_linalg.config import load_config
from vector_linalg.embeddings import embedding_matrix, fetch_token_embeddings
from vector_linalg.interpret import print_results
from vector_linalg.metrics import results_table
from vector_linalg.plots import figure_distance_error, figure_recall_vs_bits, figure_token_pca
from vector_linalg.study import run_compression_study

cfg = load_config()

## 1. Token embeddings (data)

In [ ]:
df = fetch_token_embeddings(cfg)
tokens, keys = embedding_matrix(df, cfg.glove_dim)
display(Markdown(f"**{len(tokens)}** tokens, **d={cfg.glove_dim}** (GloVe)"))
display(df.head(8))

## 2. Embedding geometry (PCA)

In [ ]:
fig = figure_token_pca(keys, tokens, highlight=["king", "queen", "man", "woman", "paris", "france"])
display(fig)

## 3. Compression study

Methods: full precision, JL sketch, rank-k SVD, 1-bit signs, scalar quantization.

Metric: **recall@k** — does compressed scoring find the same nearest token neighbors?

In [ ]:
results = run_compression_study(keys, cfg)
display(results_table(results, cfg.recall_k))

best = max(results, key=lambda r: r.recall_at_k)
display(Markdown(
    f"**Best recall@{cfg.recall_k}:** `{best.method}` = {best.recall_at_k:.3f} "
    f"at ~{best.bits_per_dim:.2f} bits/dim"
))

## 4. Figures

In [ ]:
display(figure_recall_vs_bits(results, cfg.recall_k))
display(figure_distance_error(results))

## 5. Takeaway

Compression is a **geometry preservation** problem. Extreme methods (1-bit signs, low-rank) save memory but hurt recall unless paired with correction — the motivation for multi-stage pipelines like TurboQuant (spectral/polar + JL residual).